# nb_09 — FactInvoice & FactInvoiceLine

**Purpose:** Build `FactInvoice` and `FactInvoiceLine` by joining invoice CSVs to dimension surrogate keys.

## Overview
- Reads `fact_invoice_header.csv` and `fact_invoice_lines.csv`
- Resolves surrogate keys from: DimCustomer, DimSupplier, DimDate (×3 date roles), DimProduct
- Writes `FactInvoice` and `FactInvoiceLine` Delta tables
- Runs OPTIMIZE + ZORDER on both tables

**Prerequisites:** nb_02 through nb_06 must be run first.

In [ ]:
from pyspark.sql.functions import col, to_date, date_format, when

# Load dimension tables (current records only for SCD2 dims)
dim_customer   = spark.table("DimCustomer").filter(col("is_current") == True).select("customer_id", "customer_key")
dim_supplier   = spark.table("DimSupplier").filter(col("is_current") == True).select("supplier_id", "supplier_id".replace("supplier_id", "supplier_key") if False else "supplier_id")

# Re-read cleanly
dim_customer = spark.sql("SELECT customer_id, customer_key FROM DimCustomer WHERE is_current = true")
dim_supplier = spark.sql("SELECT supplier_id FROM DimSupplier WHERE is_current = true")
dim_date     = spark.sql("SELECT date_key, CAST(CONCAT(SUBSTR(CAST(date_key AS STRING),1,4),'-',SUBSTR(CAST(date_key AS STRING),5,2),'-',SUBSTR(CAST(date_key AS STRING),7,2)) AS DATE) AS cal_date FROM DimDate")

print("Dimensions loaded.")

In [ ]:
# Helper: date string -> date_key INT
from pyspark.sql.functions import regexp_replace, concat, substring, lpad

def date_col_to_key(df, date_col, key_col):
    """Convert a date/string column to a YYYYMMDD integer surrogate key."""
    return df.join(
        dim_date.withColumnRenamed("cal_date", date_col).withColumnRenamed("date_key", key_col),
        on=date_col,
        how="left"
    )

# Read fact header
df_header = spark.read.csv(
    "Files/seed_data/fact_invoice_header.csv",
    header=True, inferSchema=True
)
print(f"Invoice headers: {df_header.count()}")
df_header.printSchema()

In [ ]:
# Cast date columns and look up date keys
from pyspark.sql.functions import to_date, lit

df_header = (
    df_header
    .withColumn("invoice_date_parsed", to_date(col("invoice_date")))
    .withColumn("due_date_parsed",     to_date(col("due_date")))
    .withColumn("payment_date_parsed", to_date(col("payment_date")))
)

# Build date map: cal_date -> date_key
date_map = spark.sql("""
    SELECT date_key,
           CAST(CONCAT(SUBSTR(CAST(date_key AS STRING),1,4),'-',
                       SUBSTR(CAST(date_key AS STRING),5,2),'-',
                       SUBSTR(CAST(date_key AS STRING),7,2)) AS DATE) AS cal_date
    FROM DimDate
""")

df_inv = (
    df_header
    .join(date_map.withColumnRenamed("cal_date","invoice_date_parsed").withColumnRenamed("date_key","invoice_date_key"), on="invoice_date_parsed", how="left")
    .join(date_map.withColumnRenamed("cal_date","due_date_parsed").withColumnRenamed("date_key","due_date_key"),     on="due_date_parsed",     how="left")
    .join(date_map.withColumnRenamed("cal_date","payment_date_parsed").withColumnRenamed("date_key","payment_date_key"), on="payment_date_parsed", how="left")
    .join(dim_customer, on="customer_id", how="left")
)

print("Header joins complete.")

In [ ]:
# Write FactInvoice
(
    df_inv.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("FactInvoice")
)

spark.sql("OPTIMIZE FactInvoice ZORDER BY (invoice_date_key, customer_key)")
print("FactInvoice written and optimized.")
spark.sql("SELECT COUNT(*) AS row_count FROM FactInvoice").show()

In [ ]:
# Read and write FactInvoiceLine
df_lines = spark.read.csv(
    "Files/seed_data/fact_invoice_lines.csv",
    header=True, inferSchema=True
)

# Join DimProduct surrogate key
dim_product = spark.sql("SELECT product_id, product_key FROM DimProduct WHERE is_current = true")
df_lines = df_lines.join(dim_product, on="product_id", how="left")

(
    df_lines.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("FactInvoiceLine")
)

spark.sql("OPTIMIZE FactInvoiceLine ZORDER BY (invoice_id, product_key)")
print("FactInvoiceLine written and optimized.")
spark.sql("SELECT COUNT(*) AS row_count FROM FactInvoiceLine").show()